# CTA多资产多空策略：研究入口

研究从37只ETF的日线行情、市场波动率状态和EFFR现金基准出发，按输入核验、逐月预测、Validation选择、五日滚动组合和历史Test展示推进。原始输入由运行者在本地提供。Training的组合参数来自后续Validation，因此Training账户曲线用于事后参数诊断。部分资产缺价时，受影响的完整五日窗口从历史账户中排除，其余可用资产继续参与；完整窗口的判断使用事后行情。

## 1．研究环境与统一设置

`Config`集中定义样本阶段、六项模型特征、五日持仓与一日执行延迟、单资产权重、组合目标敞口及参数候选。工程入口加载本地输入后，研究对象和组合参数网格随数据质量信息一并呈现。

In [ ]:
from pathlib import Path
import sys
from IPython.display import Image, display

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'pyproject.toml').exists()), None)
if root is None:
    raise FileNotFoundError('请从工程根目录或其notebooks子目录启动Notebook。')
if not (root / 'data' / 'raw' / '_manifest.csv').exists():
    raise FileNotFoundError('请按data/INPUTS.md在data/raw目录提供本地研究输入。')
sys.path.insert(0, str(root / 'src'))

from cta_project.config import Config, FEATURES, MODEL_NAMES, PARAMETER_VALUES
from cta_project.pipeline import ResearchProject
from cta_project.reporting import save_figures
from cta_project.storage import input_manifest

config = Config.in_project(root)
project = ResearchProject(config)
print('工程：', root)
print('资产数：', len(project.market.symbols))
print('市场交易日：', len(project.market.calendar))
print('部分资产缺价的交易日：', int(project.market.close.isna().any(axis=1).sum()))
print('特征：', FEATURES)
print('模型：', MODEL_NAMES)
print('组合参数网格：', len(PARAMETER_VALUES), '×', len(PARAMETER_VALUES))

## 2．输入数据、缺价与标签时间

市场日历保留个别ETF缺价的交易日，价格拆分调整在内存中完成。信号日为$t$时，次日收盘执行、持有五个市场交易日的目标收益为$y_{i,t}^{(5)}=P_{i,t+6}/P_{i,t+1}-1$。模型训练只使用在预测月之前已经成熟的标签。完整五日窗口筛选属于事后历史研究口径。

In [ ]:
display(project.market.quality)
print('输入与代码指纹：', input_manifest(config)['fingerprint'])
print('训练、验证、历史测试区间：', config.stages)
print('标签终点缺失数：', int(project.market.label_end.isna().sum()))

## 3．逐月预测与全量研究

每个预测月先用此前已成熟的数据选择MACD参数，再依三个月拟合、一个月局部验证、下一个月预测的顺序训练七类模型。资产信号依次经过历史缩尾、横截面归一化和训练窗标准化；`VOL_STATE`的横截面归一化口径与保留VIX／MOVE历史分位数信息的候选口径在共同Validation样本上比较。模型预测之后，优化器为每日新建的五日持仓批次决定可用ETF的多空方向及目标权重。确定性多起点求解在可行候选中选择目标函数值最低者，不保证全局最优。

完整研究从逐月预测延伸至账户核算，结果与图像由同一运行过程保存。逐月预测和每模型14×14组组合参数按批次缓存。

In [ ]:
# 全量研究可在本Notebook中运行，也可在工程根目录执行命令行入口。
summary = project.run_full(workers=4)
figures = save_figures(config, summary)
print('Validation选定模型：', summary['selected_model'])
print('Validation选定的VOL_STATE口径：', summary['selected_variant'])
print('结果目录：', config.results_dir)

## 4．Validation中的特征、模型与组合选择

研究先以成熟的Validation五日收益标签评价MSE和MAE；两项误差均处于各自最优值0.5％以内的模型进入组合比较。每个模型的`λ`与`γ`来自相同的14×14网格，组合在回撤门槛下按扣费后收益及回撤排序。目标波动率和高波动状态乘数同样只依据Validation选定；历史Test留作已选方案的展示。

In [ ]:
display(summary['feature_choice'])
display(summary['joint'])
display(summary['choices'])
display(summary['risk_choice'])

## 5．残差协方差的同决策日诊断

主线组合使用逐月滚动训练窗内的拟合残差，经Ledoit–Wolf收缩估计协方差。另一个估计口径使用此前逐月样本外预测的已成熟残差。固定模型、`λ`、`γ`和Validation决策日后，对照结果同时呈现收益、回撤与窗口覆盖情况。此诊断不进入最终账户选择。

In [ ]:
display(summary['covariance_windows'])
display(summary['covariance_comparison'])
candidate_figure = figures / 'validation_covariance_candidate.png'
if candidate_figure.exists():
    display(Image(filename=str(candidate_figure)))

## 6．三阶段账户、实际敞口与投资基准

优化器约束每批五日目标仓位：单资产−10％至＋20％、净敞口90％至100％、总敞口280％至300％。五批重叠后的账户实际敞口另行统计。成交佣金与卖出端Section 31费用在执行日进入现金；完整阶段与首笔成交日起的有效交易期分别展示几何年化收益、波动率、回撤、Sharpe比率和换手率。

图表依次呈现Training、Validation和历史Test的七模型账户、风险控制对照，以及选定策略与37只ETF等权买入并持有基准。基准在首笔策略成交日等额买入、期末卖出，采用相同费用代理；个别ETF缺价的日期不展示基准净值。Training预测逐月只用既往信息，其组合参数却由后续Validation选定，因此该段组合绩效为事后参数诊断。分红、融资、借券和滑点未计入价格收益账户。

In [ ]:
display(summary['three_stage'])
display(summary['benchmark_summary'])
display(summary['selected_risk_summary'])
display(summary['risk_summary'])
for stage in ('Training', 'Validation', '历史Test'):
    print(stage, '：七模型')
    display(Image(filename=str(figures / f'{stage}_seven_models.png')))
    print(stage, '：组合风险控制')
    display(Image(filename=str(figures / f'{stage}_risk_control.png')))
    print(stage, '：策略与等权买入并持有')
    display(Image(filename=str(figures / f'{stage}_buy_hold_benchmark.png')))

## 7．结果与复现

运行清单记录本地输入、源码、配置及软件环境的SHA256指纹；逐月预测、组合参数网格、三阶段结果与图像保存在`results/`。相同指纹的运行可复用已完成批次。数据与费用口径见`docs/PROVENANCE.md`，研究方法见`docs/ARCHITECTURE.md`，测试与复现方法见`docs/TEST_REPORT.md`。

In [ ]:
print('运行清单：', config.results_dir / 'run_manifest.json')
print('组合网格目录：', config.results_dir / 'portfolio_grids')
print('图像目录：', figures)